# 12.5 - LangChain Memory
**Phase:** 12 - LangChain / Framework Abstractions
**Status:** VERIFIED
---
## 1. What Are We Solving?
LLMs are stateless: each call is independent. Memory (message history) lets a model reference
earlier turns in a multi-turn conversation.
## 2. Why Does This Matter?
Without memory a chatbot forgets the user's name every message. Memory is the bridge from
single-shot pipelines to interactive assistants.
## 3. Prerequisites
- Unit 12.4 (chains)
- Chat roles system/user/assistant; `MessagesPlaceholder`
## 4. Learning Objectives
By the end of this unit, you should be able to:
- Store per-session history with `InMemoryChatMessageHistory`
- Wrap a chain with `RunnableWithMessageHistory`
- Isolate history across threads/sessions
- Add a custom counter via a simple loader/saver
- Trim history to bound its size
## 5. Mental Model
Memory is a notebook the LLM can read. Each user message and assistant reply get appended; the model
sees the full notebook (within a trim window), not just the latest question.

```text
user turns --append--> InMemoryChatMessageHistory (per session_id)
                          ^
                          | loaded each turn
                    RunnableWithMessageHistory -> prompt(MessagesPlaceholder) -> model


## 6. Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


In [2]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage, HumanMessage
from langchain_groq import ChatGroq


def chat_invoke(messages, temperature=0.0, model=GROQ_MODEL):
    if not os.environ.get("GROQ_API_KEY"):
        n_turns = 0
        for m in messages:
            t = getattr(m, "type", None)
            if t is None and isinstance(m, tuple):
                t = m[0]
            if t in ("human", "ai"):
                n_turns += 1
        return AIMessage(content=f"mock: saw {n_turns} conversational turns in history")
    try:
        return ChatGroq(model=model, temperature=temperature).invoke(messages)
    except Exception as e:
        return AIMessage(content=f"[llm-error: {type(e).__name__}]")


safe_model = RunnableLambda(lambda m: chat_invoke(m))
store = {}   # session_id -> InMemoryChatMessageHistory


def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the history to stay consistent."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{input}"),
])

chain = prompt | safe_model
with_history = RunnableWithMessageHistory(
    chain, get_history,
    input_messages_key="input",
    history_messages_key="history",
)
print("RunnableWithMessageHistory ready")


RunnableWithMessageHistory ready


D:\CODE\complete ml\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 7. Three Turns, One Thread — History Accumulates
Turn one stores the assistant reply; the next turn's prompt already contains the prior messages.

In [3]:
def show_state(history):
    print("  messages in store:", len(history.messages))
    for m in history.messages:
        print(f"    {m.type:8}: {m.content[:40]}")


cfg = {"configurable": {"session_id": "thread-A"}}
for i, q in enumerate(["My name is Ada.",
                       "What is my name?",
                       "What did I tell you first?"]):
    out = with_history.invoke({"input": q}, cfg)
    print(f"turn {i+1}: question={q!r}")
    print("  reply :", out.content)
show_state(store["thread-A"])


turn 1: question='My name is Ada.'
  reply : Nice to meet you, Ada! How can I help you today?


turn 2: question='What is my name?'
  reply : Your name is Ada.


turn 3: question='What did I tell you first?'
  reply : You first told me, “My name is Ada.”
  messages in store: 6
    human   : My name is Ada.
    ai      : Nice to meet you, Ada! How can I help yo
    human   : What is my name?
    ai      : Your name is Ada.
    human   : What did I tell you first?
    ai      : You first told me, “My name is Ada.”


## 8. Two Threads Stay Isolated
Session IDs prevent cross-user leakage. Thread B has its own notebook and does not see thread A.

In [4]:
cfg_b = {"configurable": {"session_id": "thread-B"}}
out_b = with_history.invoke({"input": "My favorite color is teal."}, cfg_b)
print("B reply:", out_b.content)
print()
print("thread-A history:")
show_state(store["thread-A"])
print("thread-B history:")
show_state(store["thread-B"])


B reply: That’s a great choice! Teal is such a versatile color—cool and calming, yet vibrant enough to make a statement. Do you have a favorite shade of teal, or is it the general hue that you love?

thread-A history:
  messages in store: 6
    human   : My name is Ada.
    ai      : Nice to meet you, Ada! How can I help yo
    human   : What is my name?
    ai      : Your name is Ada.
    human   : What did I tell you first?
    ai      : You first told me, “My name is Ada.”
thread-B history:
  messages in store: 2
    human   : My favorite color is teal.
    ai      : That’s a great choice! Teal is such a ve


## 9. Printed Accumulated Context
Show what the model actually receives each turn: the formatted prompt with the history injected into
the placeholder.

In [5]:
final_msgs = with_history.with_config({"configurable": {"session_id": "thread-A"}})
formatted = prompt.invoke({"history": store["thread-A"].messages, "input": "remind me"})
print("Prompt sent to the model contains history placeholder filled in:")
for m in formatted.to_messages():
    print(f"  [{m.type}] {m.content[:50]}")


Prompt sent to the model contains history placeholder filled in:
  [system] You are a helpful assistant. Use the history to st
  [human] My name is Ada.
  [ai] Nice to meet you, Ada! How can I help you today?
  [human] What is my name?
  [ai] Your name is Ada.
  [human] What did I tell you first?
  [ai] You first told me, “My name is Ada.”
  [human] remind me


## 10. SimpleMemory: A Custom Loader/Saver With a Counter
In LangChain a memory has `load_memory_variables` (loader) and `save_context` (saver). We build a
tiny one that counts how many times each session writes, demonstrating that "memory" is just a
pair of functions you control.

In [6]:
class SimpleMemory:
    """A minimal loader/saver that also counts writes per session."""
    def __init__(self):
        self.data = {}        # session_id -> {"history": [], "writes": 0}
    def load_memory_variables(self, session_id: str) -> dict:
        d = self.data.setdefault(session_id, {"history": [], "writes": 0})
        return {"history": list(d["history"])}
    def save_context(self, session_id: str, user: str, ai: str):
        d = self.data.setdefault(session_id, {"history": [], "writes": 0})
        d["history"] += [("user", user), ("assistant", ai)]
        d["writes"] += 1


mem = SimpleMemory()
for q, a in [("hi", "hello!"), ("how are you", "great, you?"), ("bye", "see you")]:
    mem.save_context("s1", q, a)
print("loaded vars for s1:", mem.load_memory_variables("s1"))
print("total writes for s1:", mem.data["s1"]["writes"])


loaded vars for s1: {'history': [('user', 'hi'), ('assistant', 'hello!'), ('user', 'how are you'), ('assistant', 'great, you?'), ('user', 'bye'), ('assistant', 'see you')]}
total writes for s1: 3


## 11. Trim / Limit Demonstration
Unbounded history explodes token cost. We cap a session's stored messages to the most recent N by
trimming the underlying history list after each turn.

In [7]:
def get_history_trimmed(session_id: str, max_messages: int = 4):
    h = get_history(session_id)
    if len(h.messages) > max_messages:
        h.messages = h.messages[-max_messages:]
    return h


nogrow_prompt = ChatPromptTemplate.from_messages([
    ("system", "Be brief."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{input}"),
])
with_trim = RunnableWithMessageHistory(
    nogrow_prompt | safe_model, get_history_trimmed,
    input_messages_key="input", history_messages_key="history",
)
cfg_t = {"configurable": {"session_id": "thread-T", "max_messages": 4}}
for i in range(6):
    with_trim.invoke({"input": f"message number {i}"}, cfg_t)
get_history_trimmed("thread-T", 4)
print("stored messages after 6 turns (cap=4):", len(store["thread-T"].messages))
print("kept (tail):")
for m in store["thread-T"].messages:
    print(f"  [{m.type}] {m.content[:40]}")


D:\CODE\complete ml\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


stored messages after 6 turns (cap=4): 4
kept (tail):
  [human] message number 4
  [ai] Message 4 received.
  [human] message number 5
  [ai] Message 5 received.




## Common Mistakes

- (3-5 bullets, from roadmap, concrete and specific to the unit)

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| ... | ... | ... |
(row table, 3-5 rows)

## Best Practices

- (3-5 bullets)

## Hands-On Practice

1. **Basic:** ...
2. **Guided:** ...
3. **Independent:** ...
4. **Realistic:** ...
5. **Challenge:** ...

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.


### Common Mistakes (applied)

- Growing history without bound (token cost explodes) — trim it.
- No session IDs (all users share one notebook).
- Forgetting the `MessagesPlaceholder` so history never reaches the prompt.
- Using `ConversationBufferMemory`/`ConversationSummaryMemory` legacy classes where
  `RunnableWithMessageHistory` is the modern approach.

### Debugging (applied)

| Symptom | Likely Cause | Fix |
|---|---|---|
| Model forgets prior turns | History not in the prompt | Add `MessagesPlaceholder(variable_name="history")` |
| Token count explodes | Full history un-trimmed | Cap `max_messages` |
| Sessions share history | Missing/duplicate session id | Unique `session_id` per user |
| Duplicate messages | Two append points | Keep one append point |

### Best Practices (applied)

- Always use session IDs for multi-user systems.
- Trim or summarise long conversations.
- Test memory across 5+ turns.

### Hands-On Practice

1. **Basic:** Run 5 turns in one thread; observe the growing notebook.
2. **Guided:** Make the assistant remember the user's name across turns.
3. **Independent:** Raise `max_messages` and watch the count; verify trimming keeps the tail.
4. **Realistic:** Two threads, distinct questions, verify zero leakage.
5. **Challenge:** Compare full-history vs trimmed-history token usage for a 20-turn run.

### Exit Criteria

- You can build multi-turn conversations with memory.
- You can choose the right memory type for a given use case.
- You can handle session isolation.
